# 04 - Model Evaluation

Цель: показать classification metrics, confusion matrix, ROC/PR curves.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"
FIGURES = REPORTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print("Project root:", ROOT)


Project root: /Users/andrey/Documents/projects/COMPASS-AI


In [ ]:
from sklearn.metrics import auc, confusion_matrix, precision_recall_curve, roc_curve

metrics_path = REPORTS / "model_metrics.json"
ranking_path = REPORTS / "ranking_metrics.json"
predictions_path = REPORTS / "test_predictions.csv"

with open(metrics_path, encoding="utf-8") as file:
    model_metrics = json.load(file)

with open(ranking_path, encoding="utf-8") as file:
    ranking_metrics = json.load(file)

predictions = pd.read_csv(predictions_path)

display(pd.DataFrame([model_metrics]))


,accuracy,precision,recall,f1,roc_auc,pr_auc,positive_rate,predicted_positive_rate,rows
0,0.78104,0.775764,0.733595,0.75409,0.859216,0.820082,0.457642,0.432766,8924.0


In [3]:
y_true = predictions["success_label"]
y_pred = predictions["prediction"]
y_score = predictions["success_probability"]

cm = confusion_matrix(y_true, y_pred)
cm_df = pd.DataFrame(
    cm,
    index=["actual_0", "actual_1"],
    columns=["pred_0", "pred_1"],
)

display(cm_df)


,pred_0,pred_1
actual_0,3974,866
actual_1,1088,2996


In [4]:
import plotly.express as px

fig = px.imshow(cm_df, text_auto=True)
fig.update_layout(title="Confusion matrix")
fig.show()
fig.write_html(FIGURES / "evaluation_confusion_matrix.html")


In [5]:
import plotly.express as px

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
roc_df = pd.DataFrame({"fpr": fpr, "tpr": tpr})

fig = px.line(roc_df, x="fpr", y="tpr")
fig.update_layout(title=f"ROC curve AUC={roc_auc:.4f}")
fig.show()
fig.write_html(FIGURES / "evaluation_roc_curve.html")


In [6]:
import plotly.express as px

precision, recall, _ = precision_recall_curve(y_true, y_score)
pr_df = pd.DataFrame({"precision": precision, "recall": recall})

fig = px.line(pr_df, x="recall", y="precision")
fig.update_layout(title="Precision-Recall curve")
fig.show()
fig.write_html(FIGURES / "evaluation_pr_curve.html")


In [7]:
import plotly.express as px

ranking_df = (
    pd.DataFrame(ranking_metrics)
    .T
    .reset_index()
    .rename(columns={"index": "model"})
)
display(ranking_df)

fig = px.bar(
    ranking_df,
    x="model",
    y=["precision_at_1", "precision_at_3", "ndcg_at_3", "mrr"],
    barmode="group",
)
fig.update_layout(title="Ranking metrics comparison")
fig.show()
fig.write_html(FIGURES / "evaluation_ranking_metrics.html")


,model,precision_at_1,precision_at_3,ndcg_at_3,mrr,tasks_evaluated,rows_evaluated
0,ml_model,0.893333,0.853333,0.862555,0.943778,375.0,8924.0
1,rule_based_baseline,0.872000,0.859556,0.861110,0.932667,375.0,8924.0
2,random_baseline,0.485333,0.501333,0.495777,0.667997,375.0,8924.0
